# RQ2 and RQ3 analyses: own-source thresholds, image properties, matched subsets
Three analyses from saved predictions and cached image properties. **No model is trained and no test set is opened again.** All of it is **post-hoc** and is labelled that way in the thesis.

1. **Own-source operating points (RQ2).** The frozen threshold from DDR validation does not transfer. How much does a threshold fitted on the target source itself recover?
2. **Image properties (RQ3).** (a) Can image properties (size, file size, brightness, colour, sharpness) tell *which dataset* an image comes from? (b) Does the model's AUROC survive inside groups of images with similar properties?
3. **Matched subsets (RQ3).** Rescore on subsets where each stratum has equal numbers of early-DR and No-DR images, so the matched properties cannot predict the label.

**Before running:** add as inputs (1) the output of your original run (`manifest.csv`, `runs/`), (2) the output of `evaluate_on_group_split` (`v2/runs/...`, preferred; the original run is used if it is missing), (3) the output of `image_property_check` (`audit/image_features.csv`). CPU only; a few minutes.

**What to send me:** the printed tables (or the CSVs in `audit/`).

In [ ]:
import glob, json, os, warnings
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
warnings.filterwarnings("ignore")
INPUT = os.environ.get("KAGGLE_INPUT", "/kaggle/input")
WORK = os.environ.get("EARLYDR_WORK", "/kaggle/working")
MIN_IMAGES = int(os.environ.get("MIN_IMAGES", 30)); MIN_EACH = int(os.environ.get("MIN_EACH", 8))
N_SAMPLE = int(os.environ.get("N_SAMPLE", 3000)); N_SPLITS = int(os.environ.get("N_SPLITS", 20)); N_MATCH = int(os.environ.get("N_MATCH", 30))
MIN_MATCH = int(os.environ.get("MIN_MATCH", 3)); MIN_DRAW = int(os.environ.get("MIN_DRAW", 20))
os.makedirs(os.path.join(WORK, "audit"), exist_ok=True)

def find(pattern):
    return glob.glob(os.path.join(INPUT, "**", pattern), recursive=True)
group_runs = sorted({f.split(os.sep + "runs" + os.sep)[0] + os.sep + "runs" for f in find("v2/runs/arm_*/seed*/preds_APTOS.csv")})
orig_runs = sorted({f.split(os.sep + "runs" + os.sep)[0] + os.sep + "runs" for f in find("runs/arm_*/seed*/preds_APTOS.csv") if os.sep + "v2" + os.sep not in f})
RUNS = (group_runs or orig_runs or [None])[0]
if RUNS is None:
    raise SystemExit("STOP: add the output of evaluate_on_group_split or of your original run (runs/arm_*/seed*/preds_*.csv).")
feature_files = find("audit/image_features.csv")
if not feature_files:
    raise SystemExit("STOP: add the output of image_property_check (audit/image_features.csv).")
print("predictions from:", RUNS, "(group split)" if group_runs else "(original run)")
print("image properties from:", feature_files[0])
rows = []
for f in glob.glob(os.path.join(RUNS, "arm_*", "seed*", "preds_*.csv")):
    seed_dir = os.path.dirname(f); arm = os.path.basename(os.path.dirname(seed_dir)); seed = int(os.path.basename(seed_dir)[4:])
    source = os.path.basename(f)[6:-4]
    d = pd.read_csv(f); d["arm"], d["seed"], d["source_set"] = arm, seed, source
    d["threshold"] = json.load(open(os.path.join(seed_dir, "calibration.json")))["threshold_dr"]
    rows.append(d)
preds = pd.concat(rows, ignore_index=True)
preds = preds[(preds.grade <= 2) & (preds.source_set != "arm_test")].copy()  # arm_test sets differ by arm, so they are left out
preds = preds.copy()                                   # early DR (1-2) vs No DR
preds["y"] = (preds.grade > 0).astype(int)
feats = pd.read_csv(feature_files[0])
print(preds.groupby(["source_set", "arm"]).seed.nunique().unstack())

## 1. Own-source operating points (post-hoc)
For each arm, seed and source, the images are split by patient into two random halves. The threshold is set at **specificity 0.90 on one half** and applied to the **other half**; this is repeated with different splits. It is compared with the frozen threshold on the same other half. Sensitivity and specificity are for early DR vs No DR.

In [ ]:
def own_vs_frozen(g):
    patients = g.patient_id.unique(); out = []
    for rep in range(N_SPLITS):
        rng = np.random.RandomState(rep); picked = set(rng.permutation(patients)[: len(patients) // 2])
        fit = g.patient_id.isin(picked).values
        if rep % 2: fit = ~fit
        ev = ~fit
        neg_fit = g.dr_score.values[fit & (g.y.values == 0)]
        if len(neg_fit) < 5 or (g.y.values[ev] == 1).sum() < 3: continue
        own = np.quantile(neg_fit, 0.90); s, y, gr = g.dr_score.values[ev], g.y.values[ev], g.grade.values[ev]
        for name, thr in [("frozen", g.threshold.iloc[0]), ("own-source", own)]:
            flag = s > thr
            out.append({"threshold": name, "sensitivity": flag[y == 1].mean(), "specificity": (~flag)[y == 0].mean(),
                        "sens_grade1": flag[gr == 1].mean() if (gr == 1).any() else np.nan})
    return pd.DataFrame(out).groupby("threshold").mean()
res = []
for (arm, seed, source), g in preds.groupby(["arm", "seed", "source_set"]):
    r = own_vs_frozen(g.reset_index(drop=True))
    for name, row in r.iterrows(): res.append({"source": source, "arm": arm, "seed": seed, "threshold": name, **row.to_dict()})
res = pd.DataFrame(res)
op = res.groupby(["source", "arm", "threshold"])[["sensitivity", "specificity", "sens_grade1"]].mean().round(3)
op.to_csv(os.path.join(WORK, "audit", "own_source_operating_points.csv"))
print(op.to_string())

## 2a. Can image properties tell which dataset an image comes from?
AUROC for telling one source from another using only the listed properties (5-fold cross-validation, no retina content). 0.5 = no difference; near 1.0 = the property alone identifies the source.

In [ ]:
FEATURE_SETS = {
    "image size (width, height, aspect)": ["width", "height", "aspect"],
    "file size": ["file_kb"],
    "colour and brightness": ["mean_b", "mean_g", "mean_r", "brightness"],
    "sharpness and field of view": ["sharpness", "retina_fraction"],
    "all properties": ["width", "height", "aspect", "file_kb", "mean_b", "mean_g", "mean_r", "brightness", "sharpness", "retina_fraction"]}
manifest_file = [f for f in find("manifest.csv")][0]
man = pd.read_csv(manifest_file)[["image_id", "source", "grade", "patient_id"]]
table = man.merge(feats, on="image_id")
table = table[table.grade <= 2]
def cv_auc(X, y):
    p = cross_val_predict(HistGradientBoostingClassifier(max_iter=150, random_state=0), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
    return roc_auc_score(y, p)
rows = []
for a, b in [("DDR", "EyePACS"), ("DDR", "APTOS"), ("EyePACS", "APTOS")]:
    sub = pd.concat([table[table.source == a].sample(min(N_SAMPLE, (table.source == a).sum()), random_state=0),
                     table[table.source == b].sample(min(N_SAMPLE, (table.source == b).sum()), random_state=0)])
    for name, cols in FEATURE_SETS.items():
        rows.append({"properties used": name, "pair": f"{a} vs {b}", "AUROC": round(cv_auc(sub[cols], (sub.source == b).astype(int)), 3)})
source_pred = pd.DataFrame(rows).pivot(index="properties used", columns="pair", values="AUROC").loc[list(FEATURE_SETS)]
source_pred.to_csv(os.path.join(WORK, "audit", "source_from_properties.csv"))
print(source_pred.to_string())

## 2b. Does the model's AUROC survive inside groups with similar properties?
Images are grouped by one property (exact image size, or eight quantile bins of brightness, sharpness, file size and redness). **Within-group AUROC** uses only pairs of images from the same group, so that property cannot separate early DR from No DR. If it equals the overall AUROC on the same images, the score does not come from that property. Quantile bins control a property only coarsely.

In [ ]:
def pooled_auroc(y, s, grp):
    tot = w = 0.0
    for gname in np.unique(grp):
        m = grp == gname; pos, neg = y[m].sum(), (1 - y[m]).sum()
        if pos >= 1 and neg >= 1:
            tot += roc_auc_score(y[m], s[m]) * pos * neg; w += pos * neg
    return tot / w if w else np.nan
def groups_for(t, prop):
    if prop == "image size":
        return (t.width.astype(int).astype(str) + "x" + t.height.astype(int).astype(str)).values
    col = t.mean_r / (t.mean_r + t.mean_g + t.mean_b + 1e-6) if prop == "redness" else t[{"brightness": "brightness", "sharpness": "sharpness", "file size": "file_kb"}[prop]]
    return pd.qcut(col.rank(method="first"), 8, labels=False).astype(str).values
pf = preds.merge(feats, on="image_id")
out = []
for (source, arm, seed), t in pf.groupby(["source_set", "arm", "seed"]):
    y, s = t.y.values, t.dr_score.values
    for prop in ["image size", "brightness", "sharpness", "file size", "redness"]:
        grp = groups_for(t, prop)
        stats = pd.DataFrame({"g": grp, "y": y}).groupby("g").y.agg(["count", "sum"])
        ok = stats.index[(stats["count"] >= MIN_IMAGES) & (stats["sum"] >= MIN_EACH) & (stats["count"] - stats["sum"] >= MIN_EACH)]
        keep = np.isin(grp, ok)
        if len(ok) < 2 or len(np.unique(y[keep])) < 2: continue
        out.append({"source": source, "arm": arm, "seed": seed, "grouped by": prop, "usable groups": len(ok), "images used": int(keep.sum()),
                    "overall AUROC": roc_auc_score(y[keep], s[keep]), "within-group AUROC": pooled_auroc(y[keep], s[keep], grp[keep])})
strata = pd.DataFrame(out)
if strata.empty: raise SystemExit("No property group was large enough; lower MIN_IMAGES / MIN_EACH at the top of the first cell.")
agg = strata.groupby(["source", "grouped by", "arm"])[["usable groups", "images used", "overall AUROC", "within-group AUROC"]].mean()
agg["drop"] = agg["overall AUROC"] - agg["within-group AUROC"]
agg = agg.round(3); agg.to_csv(os.path.join(WORK, "audit", "property_strata.csv"))
print(agg.to_string())

## 3. Matched subsets
Within each stratum (same image size, optionally also the same brightness and sharpness tercile) a random subset with **equal numbers of early-DR and No-DR images** is drawn, repeated many times. The model's AUROC on these subsets, with the 2.5-97.5 percentile range over repeats, shows what is left once the matched properties carry no label information. **Property-only AUROC** is a check that matching worked (about 0.5 means the properties no longer predict the label).

In [ ]:
ALL_PROPS = FEATURE_SETS["all properties"]
def tercile(col): return pd.qcut(col.rank(method="first"), 3, labels=False).astype(str)
def strata_key(t, level):
    key = t.width.astype(int).astype(str) + "x" + t.height.astype(int).astype(str)
    if level >= 2: key = key + "|b" + tercile(t.brightness)
    if level >= 3: key = key + "|s" + tercile(t.sharpness)
    return key.values
def matched_indices(key, y, rng):
    idx = []
    for k in np.unique(key):
        m = np.where(key == k)[0]; pos, neg = m[y[m] == 1], m[y[m] == 0]; n = min(len(pos), len(neg))
        if n >= MIN_MATCH: idx += list(rng.choice(pos, n, replace=False)) + list(rng.choice(neg, n, replace=False))
    return np.array(idx)
LEVELS = {1: "same image size", 2: "size + brightness tercile", 3: "size + brightness + sharpness terciles"}
out = []
for (source, arm), t_all in pf.groupby(["source_set", "arm"]):
    per_seed = []
    for seed, t in t_all.groupby("seed"):
        t = t.reset_index(drop=True); y, s = t.y.values, t.dr_score.values
        for level, label in LEVELS.items():
            key = strata_key(t, level); vals = []
            for rep in range(N_MATCH):
                idx = matched_indices(key, y, np.random.RandomState(rep))
                if len(idx) >= MIN_DRAW and len(np.unique(y[idx])) == 2: vals.append(roc_auc_score(y[idx], s[idx]))
            if vals: per_seed.append({"level": label, "seed": seed, "mean": np.mean(vals), "lo": np.percentile(vals, 2.5), "hi": np.percentile(vals, 97.5), "n": len(idx)})
    ps = pd.DataFrame(per_seed)
    if ps.empty: print(f"{source} / {arm}: no stratum large enough for matching"); continue
    for label, g in ps.groupby("level"):
        out.append({"source": source, "arm": arm, "matched on": label, "images per draw": int(g.n.mean()),
                    "matched AUROC": round(g["mean"].mean(), 3), "range (2.5-97.5%)": f"{g.lo.mean():.3f}-{g.hi.mean():.3f}"})
matched = pd.DataFrame(out)
# did matching remove the property signal? property-only AUROC on one matched draw (arm-independent)
chk = []
base = pf[pf.arm == pf.arm.iloc[0]]; base = base[base.seed == base.seed.iloc[0]]
for source, t in base.groupby("source_set"):
    t = t.reset_index(drop=True)
    for level, label in LEVELS.items():
        idx = matched_indices(strata_key(t, level), t.y.values, np.random.RandomState(0))
        if len(idx) >= MIN_DRAW and len(np.unique(t.y.values[idx])) == 2:
            sub = t.iloc[idx]; chk.append({"source": source, "matched on": label, "property-only AUROC (should be near 0.5)": round(cv_auc(sub[ALL_PROPS], sub.y), 3)})
chk = pd.DataFrame(chk)
matched.to_csv(os.path.join(WORK, "audit", "matched_subsets.csv"), index=False); chk.to_csv(os.path.join(WORK, "audit", "matched_property_check.csv"), index=False)
print(matched.to_string(index=False)); print("\n", chk.to_string(index=False))

## How to read the results
* **Own-source thresholds:** if specificity returns to about 0.90 and sensitivity stays high, the transfer failure was only the operating point (a calibration problem), not the ranking.
* **Source from properties:** high AUROC means that property alone separates the datasets, so a merged model could learn the source from it. File size and sharpness also rise with lesion content, so they are weaker camera proxies than image size.
* **Strata and matched AUROC:** a small drop from overall to within-group (or matched) AUROC means the score does not depend on that property. A large drop means the model may use it.
* Camera model is not recorded in these datasets: every property here is a proxy.